In [21]:
import pynq
import time
import numpy as np

In [2]:
bitstream = "/home/xilinx/design_1.bit"
overlay = pynq.Overlay(bitstream)

In [22]:
overlay?

In [23]:
dut = overlay.matmul_0

In [24]:
dut?

In [32]:
A = pynq.allocate((100, 150), dtype='int16')
B = pynq.allocate((150, 200), dtype='int16')
C = pynq.allocate((100, 200), dtype='int16')

In [33]:
np.random.seed(0) 
A[:] = np.random.randint(0, 256, size=A.shape, dtype='int16') 
B[:] = np.random.randint(0, 256, size=B.shape, dtype='int16')

In [34]:
A.sync_to_device()
B.sync_to_device()

In [ ]:
dut.register_map

In [36]:
dut.register_map.Matrix_A_DRAM_1 = A.device_address & 0xFFFFFFFF
dut.register_map.Matrix_A_DRAM_2 = A.device_address >> 32

dut.register_map.Matrix_B_DRAM_1 = B.device_address & 0xFFFFFFFF
dut.register_map.Matrix_B_DRAM_2 = B.device_address >> 32

dut.register_map.Matrix_C_DRAM_1 = C.device_address & 0xFFFFFFFF
dut.register_map.Matrix_C_DRAM_2 = C.device_address >> 32

In [ ]:
start_time = time.time()
dut.register_map.CTRL.AP_START = 1
dut.register_map.CTRL[4] = 1
while not dut.register_map.CTRL.AP_DONE: 
    pass
end_time = time.time()
duration = end_time - start_time

print(f'Kernel completed in {duration * 1000:.2f}ms')

In [38]:
C.sync_from_device()

In [ ]:
C_cpu = np.matmul(A, B)

if np.allclose(C, C_cpu, atol=1e-5):
    print("FPGA output matches CPU output! Verification successful.")
else:
    print("FPGA output does not match CPU output. Verification failed.")

In [40]:
overlay.free()

In [ ]:
start_time = time.time()
c_host = np.matmul(A, B)
end_time = time.time()
duration = end_time - start_time

print(f'PS computation completed in {duration * 1000:.2f}ms')